# Entrenamiento MelanomaAI en Google Colab

### Pasos previos (hacer UNA sola vez)
1. Subir la carpeta del proyecto a Google Drive en la ruta:
   ```
   Mi unidad/melanoma_ai/
   ```
   La estructura debe quedar así:
   ```
   melanoma_ai/
   ├── train.py
   ├── dataset_aumentado/   ← carpetas con imágenes
   └── checkpoints/         ← se crea automáticamente
   ```
2. En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**
3. Ejecutar las celdas en orden.

In [ ]:
# ── Celda 1: Verificar GPU ────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  Sin GPU — activa T4 en Entorno de ejecución')

In [ ]:
# ── Celda 2: Montar Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/melanoma_ai'
os.chdir(PROJECT_DIR)
print('Directorio actual:', os.getcwd())
print('Archivos:', os.listdir('.'))

In [ ]:
# ── Celda 3: Instalar / verificar dependencias ────────────────────────────
# Colab ya trae torch, torchvision y PIL preinstalados.
# Solo instalamos lo que podría faltar.
!pip install -q scikit-learn tqdm

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# ── Celda 4: Verificar dataset ────────────────────────────────────────────
from pathlib import Path

CLASSES = [
    'Extensión Superficial', 'Lentiginoso Acral', 'Lentigo Maligno',
    'Nodular', 'Otros/Mucosas', 'Otros/Oculares', 'No melanoma'
]
dataset_dir = Path('dataset_aumentado')

total = 0
for folder in sorted(dataset_dir.rglob('*')):
    if folder.is_dir():
        imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png'))
        if imgs:
            print(f'  {folder.relative_to(dataset_dir)}: {len(imgs)} imágenes')
            total += len(imgs)
print(f'\nTotal: {total} imágenes')

In [ ]:
# ── Celda 5: ENTRENAMIENTO ────────────────────────────────────────────────
#
# Parámetros recomendados para GPU T4 de Colab (15 GB VRAM):
#   --batch-size 64   → T4 aguanta 64 sin problemas
#   --workers 4       → 4 CPUs disponibles en Colab
#   --epochs 60       → suficiente con early stopping
#
# Cambia los valores según necesites.

!python train.py \
    --epochs 60 \
    --batch-size 64 \
    --workers 4 \
    --model efficientnet_b3 \
    --lr 1e-3 \
    --lr-finetune 5e-5 \
    --unfreeze-epoch 10 \
    --early-stop 15 \
    --no-melanoma-weight 2.0

In [ ]:
# ── Celda 6 (opcional): Reanudar desde checkpoint ─────────────────────────
# Útil si la sesión de Colab expiró a mitad del entrenamiento.

!python train.py \
    --resume checkpoints/best_f1.pt \
    --epochs 60 \
    --batch-size 64 \
    --workers 4 \
    --early-stop 15 \
    --no-melanoma-weight 2.0

In [ ]:
# ── Celda 7: Ver curvas de entrenamiento ──────────────────────────────────
from IPython.display import Image as IPImage, display
display(IPImage('plots/training_curves.png'))
display(IPImage('plots/confusion_matrix.png'))

In [ ]:
# ── Celda 8: Exportar a .ptl para Android ────────────────────────────────
# Solo ejecutar una vez terminado el entrenamiento y antes de cerrar Colab.
# El archivo resultante (checkpoints/melanoma_model.ptl) ya queda guardado
# en Drive automáticamente.

!python export_model.py

import os
size_mb = os.path.getsize('checkpoints/melanoma_model.ptl') / 1e6
print(f'Modelo exportado: {size_mb:.1f} MB')

In [ ]:
# ── Celda 9 (opcional): Descargar checkpoint directamente ─────────────────
# Si prefieres descargar los archivos al PC en vez de usar Drive.

from google.colab import files
files.download('checkpoints/best_f1.pt')
files.download('checkpoints/melanoma_model.ptl')